In [30]:
import json
import os
import shutil
import re

In [28]:
def process_node(node):
    if 'raw_text' in node:
        return node['raw_text'].strip()

    if 'AND' in node:
        left_text = process_node(node['AND']['left'])
        right_text = process_node(node['AND']['right'])
        return combine_texts(left_text, right_text, '[AND]')

    if 'OR' in node:
        left_text = process_node(node['OR']['left'])
        right_text = process_node(node['OR']['right'])
        return combine_texts(left_text, right_text, '[OR]')

    if 'NOT' in node:
        left_text = process_node(node['NOT']['left'])
        return f"[NOT] {left_text}"

    return ""

def combine_texts(left_text, right_text, operator):
    return f"{left_text} {operator} {right_text}"

def json_to_text(json_file_path):
    with open(json_file_path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    text = process_node(data)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return '\n'.join(lines)

def process_all_files_in_directory(directory_path, output_directory, failure_directory):
    os.makedirs(output_directory, exist_ok=True)
    os.makedirs(failure_directory, exist_ok=True)

    for filename in os.listdir(directory_path):
        if filename.endswith("_exc.json") or filename.endswith("_inc.json"):
            json_file_path = os.path.join(directory_path, filename)
            try:
                output_text = json_to_text(json_file_path)
                output_file_path = os.path.join(output_directory, os.path.splitext(filename)[0] + ".txt")
                with open(output_file_path, 'w', encoding="utf-8") as output_file:
                    output_file.write(output_text)
                print(f"Processed file: {filename}")
            except Exception as e:
                print(f"Failed to process file: {filename} - Error: {e}")
                shutil.copy(json_file_path, os.path.join(failure_directory, filename))

### Json Output zu Txt Files mit Operatoren

In [29]:
# Beispielnutzung
model = "Llama3_70b_Fine-Tuned_p3_ep10_p1"
model_name = f"model_output/{model}"  # Ersetzen Sie dies durch den tatsächlichen Modellnamen
input_directory_path = os.path.join(model_name, "output")
output_directory_path = os.path.join(model_name, "processed_output")
failure_directory_path = os.path.join(model_name, "failure")
process_all_files_in_directory(input_directory_path, output_directory_path, failure_directory_path)

Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860220_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860259_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860311_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860324_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860363_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860402_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860402_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860428_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860493_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860545_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860597_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860610_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860623_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860662_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03860688_exc.

In [31]:
ic = "Inclusion Criteria:"
ec = "Exclusion Criteria:"

In [32]:
def combine_files(input_dir, output_dir):
    def read_file(filepath):
        with open(filepath, 'r', encoding="utf-8") as file:
            return file.read()

    def write_combined_file(nct_number, inc_text, exc_text, output_dir):
        output_filepath = os.path.join(output_dir, f'NCT{nct_number}.txt')
        with open(output_filepath, 'w', encoding="utf-8") as file:
            file.write(f"Inclusion Criteria:\n{inc_text}\n")
            file.write(f"Exclusion Criteria:\n{exc_text}\n")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    files = os.listdir(input_dir)
    nct_numbers = set()

    pattern = re.compile(r'NCT\d+')

    for file in files:
        match = pattern.search(file)
        if match:
            nct_number = match.group()
            nct_numbers.add(nct_number)
            print(f"Processing {nct_number}")

    for nct_number in nct_numbers:
        inc_files = [f for f in files if nct_number in f and '_inc' in f]
        exc_files = [f for f in files if nct_number in f and '_exc' in f]

        if inc_files and exc_files:
            inc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in inc_files)
            exc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in exc_files)
            print(inc_text)
            print(exc_text)
            write_combined_file(nct_number, inc_text, exc_text, output_dir)

In [33]:
input_dir = f'model_output/{model}/processed_output'
output_dir = f'model_output/{model}/combined_output'

combine_files(input_dir, output_dir)

Processing NCT03860220
Processing NCT03860259
Processing NCT03860311
Processing NCT03860324
Processing NCT03860363
Processing NCT03860402
Processing NCT03860402
Processing NCT03860428
Processing NCT03860493
Processing NCT03860545
Processing NCT03860597
Processing NCT03860610
Processing NCT03860623
Processing NCT03860662
Processing NCT03860688
Processing NCT03860701
Processing NCT03860753
Processing NCT03860766
Processing NCT03861078
Processing NCT03861104
Processing NCT03861195
Processing NCT03861221
Processing NCT03861312
Processing NCT03861325
Processing NCT03861377
Processing NCT03861442
Processing NCT03861507
Processing NCT03861715
Processing NCT03861715
Processing NCT03861806
Processing NCT03861845
Processing NCT03861858
Processing NCT03861962
Processing NCT03861988
Processing NCT03862027
Processing NCT03862066
Processing NCT03862105
Processing NCT03862209
Processing NCT03862209
Processing NCT03862235
Processing NCT03862274
Processing NCT03862287
Processing NCT03862404
Processing 